In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/description.md
/kaggle/input/train.csv
/kaggle/input/test.csv.zip
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/train.csv.zip
/kaggle/input/lmsys-chatbot-arena/description.md
/kaggle/input/lmsys-chatbot-arena/train.csv
/kaggle/input/lmsys-chatbot-arena/test.csv.zip
/kaggle/input/lmsys-chatbot-arena/sample_submission.csv
/kaggle/input/lmsys-chatbot-arena/sample_submission.csv.zip
/kaggle/input/lmsys-chatbot-arena/test.csv
/kaggle/input/lmsys-chatbot-arena/train.csv.zip


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss


In [3]:
# Load the data
train_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/train.csv")
test_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/test.csv")


In [4]:
train_data.head()


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,2444074745,zephyr-7b-beta,llama-2-7b-chat,"[""Can the Orca Cloud Security Platform detect ...","[""Yes, the Orca Cloud Security Platform can de...","[""Yes, the Orca Cloud Security Platform can de...",0,1,0
1,1805535695,gpt-3.5-turbo-0613,llama-2-13b-chat,"[""Write 3 sensational twists for a thriller "",...","[""1. The Protagonist's Best Friend is the Mast...","[""Sure, here are three sensational twists for ...",1,0,0
2,2454781969,claude-instant-1,gpt-4-0125-preview,"[""Create some creatively mocking sentences abo...","[""I apologize, upon further reflection I don't...","[""Sure! Just remember, this is all in good fun...",0,1,0
3,2915808846,oasst-pythia-12b,claude-instant-1,"[""what country is plaster of paris made in?""]","[""France""]","[""Plaster of Paris is not made by any single c...",0,1,0
4,3639701142,pplx-70b-online,tulu-2-dpo-70b,"[""What's the correct way to convert a potentia...","[""To correctly convert a potential undefined v...","[""In JavaScript, you can use the nullish coale...",1,0,0


In [5]:
# Combine prompt and responses
train_data['text_a'] = train_data['prompt'] + ' ' + train_data['response_a']
train_data['text_b'] = train_data['prompt'] + ' ' + train_data['response_b']
test_data['text'] = test_data['prompt'] + ' ' + test_data['response_a'] + ' ' + test_data['response_b']


In [6]:
# Prepare data for training
X = train_data[['text_a', 'text_b']]
y = train_data[['winner_model_a', 'winner_model_b', 'winner_tie']]


In [7]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:
# Vectorize the text data
vectorizer = CountVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train['text_a'] + ' ' + X_train['text_b'])
X_val_vec = vectorizer.transform(X_val['text_a'] + ' ' + X_val['text_b'])


In [9]:
# Train three separate logistic regression models
models = {}
for target in ['winner_model_a', 'winner_model_b', 'winner_tie']:
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_vec, y_train[target])
    models[target] = model


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [10]:
# Evaluate on validation set
val_predictions = {}
for target, model in models.items():
    val_predictions[target] = model.predict_proba(X_val_vec)


In [11]:
# Calculate log loss
log_losses = {}
for target, preds in val_predictions.items():
    log_losses[target] = log_loss(y_val[target], preds)


In [12]:
# Make predictions on test set
X_test_vec = vectorizer.transform(test_data['text'])
test_predictions = {}
for target, model in models.items():
    test_predictions[target] = model.predict_proba(X_test_vec)


In [13]:
# Print log loss for each target
for target, loss in log_losses.items():
    print(f"Validation Log Loss for {target}: {loss:.4f}")


Validation Log Loss for winner_model_a: 0.7646
Validation Log Loss for winner_model_b: 0.7669
Validation Log Loss for winner_tie: 0.7327


In [14]:
# Load the test data
test_data = pd.read_csv("/kaggle/input/lmsys-chatbot-arena/test.csv")

# Prepare the test text data
test_data['text'] = test_data['prompt'] + ' ' + test_data['response_a'] + ' ' + test_data['response_b']

# Make predictions on the test set
test_predictions = {
    'id': test_data['id']
}

for target, model in models.items():
    # Predict probabilities for each class (0 and 1)
    probabilities = model.predict_proba(X_test_vec)
    # Take the probability of class 1 as it represents the desired output
    test_predictions[target] = probabilities[:, 1]

# Create a DataFrame from the predictions
submission_df = pd.DataFrame(test_predictions)

# Save the submission file
submission_df.to_csv("submission.csv", index=False)

print("Submission file created successfully.")


Submission file created successfully.
